HW09

In [1]:
import csv

rows = [
    ["이름", "학번", "중간", "기말", "과제"],
    ["김언어", "2026-10000", 85, 92, 90],
    ["이국문", "2026-12345", 78, 88, 85],
    ["박영문", "2026-13579", 95, 90, 100],
    ["최역사", "2025-11111", "", 82, 88],  
]

with open("scores.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(rows)

In [2]:
import json
import logging

logging.basicConfig(level=logging.INFO)

def make_report(csv_path:str, json_path:str) -> int:
    try:
        with open(csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            rows = list(reader)
    except FileNotFoundError:
        logging.warning(f"csv 파일이 존재하지 않습니다: {csv_path}")
        return 0
    except UnicodeDecodeError:
        logging.error(f"csv 파일의 인코딩이 잘못되었습니다: {csv_path}")
        return 0
    
    results = []

    for row in rows:
        name: str = row["이름"]
        sid: str = row["학번"]
        mid: int = row["중간"]
        fin: int = row["기말"]
        hw: int = row["과제"]

        if mid == "" or fin == "" or hw == "":
            mid = None if mid == "" else int(mid)
            fin = None if fin == "" else int(fin)
            hw = None if hw == "" else int(hw)
            avg = None
            grade = None
            logging.info(f"{name}: 평균 None, 등급 None (결측값 존재)")
        else:
            mid, fin, hw = int(mid), int(fin), int(hw)
            avg = round(mid*0.3 + fin*0.5 + hw*0.2, 2)
            if avg >= 90:
                grade = "A"
            elif avg >= 80:
                grade = "B"
            elif avg >= 70:
                grade = "C"
            else:
                grade = "F"
            logging.info(f"{name}: 평균 {avg}, 등급 {grade}")
        
        results.append({
            "이름": name,
            "학번": sid,
            "점수": {"중간": mid, "기말": fin, "과제": hw},
            "평균": avg,
            "등급": grade
        })

    with open(json_path, "w", encoding = "utf-8") as f:
        json.dump(results, f, ensure_ascii = False, indent = 4)

    return len(results)

make_report("scores.csv", "report.json")
with open("report.json", "r", encoding="utf-8") as f:
    print(f.read())

INFO:root:김언어: 평균 89.5, 등급 B
INFO:root:이국문: 평균 84.4, 등급 B
INFO:root:박영문: 평균 93.5, 등급 A
INFO:root:최역사: 평균 None, 등급 None (결측값 존재)


[
    {
        "이름": "김언어",
        "학번": "2026-10000",
        "점수": {
            "중간": 85,
            "기말": 92,
            "과제": 90
        },
        "평균": 89.5,
        "등급": "B"
    },
    {
        "이름": "이국문",
        "학번": "2026-12345",
        "점수": {
            "중간": 78,
            "기말": 88,
            "과제": 85
        },
        "평균": 84.4,
        "등급": "B"
    },
    {
        "이름": "박영문",
        "학번": "2026-13579",
        "점수": {
            "중간": 95,
            "기말": 90,
            "과제": 100
        },
        "평균": 93.5,
        "등급": "A"
    },
    {
        "이름": "최역사",
        "학번": "2025-11111",
        "점수": {
            "중간": null,
            "기말": 82,
            "과제": 88
        },
        "평균": null,
        "등급": null
    }
]


In [3]:
class InvalidJamoError(ValueError):
    pass

In [4]:
def classify_jamo(c:str) -> str:
    if not isinstance(c, str):
        raise TypeError(f"str 타입이 아닙니다: {type(c)}")
    if len(c) != 1:
        raise ValueError(f"길이가 1이어야 합니다: {repr(c)}")
    if "\u3131" <= c <= "\u314e":
        return "자음"
    if "\u314f" <= c <= "\u3163":
        return "모음"
    raise InvalidJamoError(f"한글 자모가 아닙니다: {repr(c)}")

In [5]:
inputs = ["ㄱ", "ㅏ", "ㄲ", "가", "AB", 5, "ㅎ", "ㅣ", ""]

for item in inputs:
    try:
        result: str = classify_jamo(item)
        print(result)
    except TypeError as e:
        print(f"[TypeError] {e}")
    except InvalidJamoError as e:
        print(f"[InvalidJamoError] {e}")
    except ValueError as e:
        print(f"[ValueError] {e}")

자음
모음
자음
[InvalidJamoError] 한글 자모가 아닙니다: '가'
[ValueError] 길이가 1이어야 합니다: 'AB'
[TypeError] str 타입이 아닙니다: <class 'int'>
자음
모음
[ValueError] 길이가 1이어야 합니다: ''


HW10

In [8]:
posts: list[str] = [
    "오늘 #파이썬 수업 진짜 재밌었음!! @prof_kim @hong 감사 ㅎㅎ "
    "자료: https://etl.snu.ac.kr/lec17",
    "@lee @park 팀플 어디서 모이지ㅠㅠ #DCCP2026 #팀플 카톡 ㄱㄱ",
    "<b>중요</b>: 다음 시험 범위는 1-15강. "
    "문의는 mam3b@snu.ac.kr (010-1234-5678)로!",
    " 여러 공백과\n\n\n줄바꿈이 많은 텍스트 ",
    "ㅋㅋㅋ #파이썬 진짜 좋다 #추천 https://snu.ac.kr",
]

In [6]:
import re
from collections import Counter

# 반복 사용 패턴 미리 컴파일
_URL     = re.compile(r'https?://\S+')
_HTML    = re.compile(r'<[^>]+>')
_EMAIL   = re.compile(r'[\w.+-]+@[\w-]+\.[a-zA-Z]{2,}')
_PHONE   = re.compile(r'\d{2,4}-\d{3,4}-\d{4}')
_MENTION = re.compile(r'@\w+')
_HASH    = re.compile(r'#\w+')
_JAMO    = re.compile(r'[ㄱ-ㅣ]+')
_SPACE   = re.compile(r'\s+')

# 해시태그 추출용 (원본에서 추출)
_HASH_EXTRACT = re.compile(r'#([가-힣a-zA-Z0-9]+)')


def clean_post(post: str) -> str:
    post = _URL.sub(' ', post)          # 1. URL → 공백
    post = _HTML.sub('', post)          # 2. HTML 태그 제거
    post = _EMAIL.sub('[이메일]', post)  # 3a. 이메일 마스킹
    post = _PHONE.sub('[전화]', post)   # 3b. 전화번호 마스킹
    post = _MENTION.sub(' ', post)      # 4a. 멘션 → 공백
    post = _HASH.sub(' ', post)         # 4b. 해시태그 → 공백
    post = _JAMO.sub('', post)          # 5. 한글 자음/모음 제거
    post = _SPACE.sub(' ', post).strip()# 6. 공백 정리
    return post


def extract_hashtags(post: str) -> list[str]:
    return _HASH_EXTRACT.findall(post)


def analyze_posts(posts: list[str]) -> dict:
    cleaned = [clean_post(p) for p in posts]
    avg_len = round(sum(len(c) for c in cleaned) / len(cleaned), 2)

    all_tags: list[str] = []
    for p in posts:
        all_tags.extend(extract_hashtags(p))
    counter = Counter(all_tags)
    hashtag_counts = dict(counter.most_common())

    masked_count = 0
    for p in posts:
        _, n1 = _EMAIL.subn('[이메일]', p)
        _, n2 = _PHONE.subn('[전화]', p)
        masked_count += n1 + n2

    return {
        "posts_n": len(posts),
        "avg_length_after_clean": avg_len,
        "hashtag_counts": hashtag_counts,
        "masked_count": masked_count,
    }

In [9]:
# clean_post 실행 결과
print("=== clean_post 결과 ===")
for i, post in enumerate(posts):
    print(f"post[{i}]: {clean_post(post)}")

# analyze_posts 실행 결과
print("\n=== analyze_posts 결과 ===")
result = analyze_posts(posts)
for key, value in result.items():
    print(f"{key}: {value}")

=== clean_post 결과 ===
post[0]: 오늘 수업 진짜 재밌었음!! 감사 자료:
post[1]: 팀플 어디서 모이지 카톡
post[2]: 중요: 다음 시험 범위는 1-15강. 문의는 [이메일].kr ([전화])로!
post[3]: 여러 공백과 줄바꿈이 많은 텍스트
post[4]: 진짜 좋다

=== analyze_posts 결과 ===
posts_n: 5
avg_length_after_clean: 20.0
hashtag_counts: {'파이썬': 2, 'DCCP2026': 1, '팀플': 1, '추천': 1}
masked_count: 2


HW12

In [10]:
import numpy as np
import pandas as pd

works = pd.DataFrame({
    '작가': ['김유정', '김유정', '김유정',
    '현진건', '현진건',
    '이상', '이상',
    '나도향', '나도향',
    '채만식', '이효석'],
    '작품': ['봄봄', '동백꽃', '만무방',
    '운수 좋은 날', 'B사감과 러브레터',
    '날개', '권태',
    '벙어리 삼룡이', '물레방아',
    '레디메이드 인생', '메밀꽃 필 무렵'],
    '연도': [1935, 1936, 1934,
    1924, 1925,
    1936, 1937,
    1925, 1925,
    1934, np.nan], # 이효석 '메밀꽃 필 무렵' 발표연도가 누락됨
    '글자수': [12500, 9800, 13600,
    11200, 7400,
    18300, 14100,
    16800, 14500,
    22000, 11800],
})

authors = pd.DataFrame({
    '작가': ['김유정', '현진건', '이상', '나도향', '채만식', '이효석', '염상섭'],
    '생년': [1908, 1900, 1910, 1902, 1902, 1907, 1897],
    '몰년': [1937, 1943, 1937, 1926, 1950, 1942, 1963],
})

In [ ]:
print(works.shape)
print(works.dtypes)
print(works.describe())

(11, 4)
작가         str
작품         str
연도     float64
글자수      int64
dtype: object
                연도           글자수
count    10.000000     11.000000
mean   1931.100000  13818.181818
std       5.546771   4080.641661
min    1924.000000   7400.000000
25%    1925.000000  11500.000000
50%    1934.000000  13600.000000
75%    1935.750000  15650.000000
max    1937.000000  22000.000000
작가     0
작품     0
연도     1
글자수    0
dtype: int64


In [13]:
print(works.isna().sum())

작가     0
작품     0
연도     1
글자수    0
dtype: int64


In [14]:
works2 = works.copy()
works2['연도'] = works2['연도'].fillna(1936).astype(int)

print(works2.dtypes)
print(works2.tail(3))

작가       str
작품       str
연도     int64
글자수    int64
dtype: object
     작가        작품    연도    글자수
8   나도향      물레방아  1925  14500
9   채만식  레디메이드 인생  1934  22000
10  이효석  메밀꽃 필 무렵  1936  11800


In [15]:
# Q2 - (a)

# (i) 1930년 이후 발표된 작품
after_1930 = works2[works2['연도'] >= 1930]
print(after_1930)

# (ii) '김유정' 또는 '이상'의 작품
kim_or_lee = works2[works2['작가'].isin(['김유정', '이상'])]
print(kim_or_lee)

     작가        작품    연도    글자수
0   김유정        봄봄  1935  12500
1   김유정       동백꽃  1936   9800
2   김유정       만무방  1934  13600
5    이상        날개  1936  18300
6    이상        권태  1937  14100
9   채만식  레디메이드 인생  1934  22000
10  이효석  메밀꽃 필 무렵  1936  11800
    작가   작품    연도    글자수
0  김유정   봄봄  1935  12500
1  김유정  동백꽃  1936   9800
2  김유정  만무방  1934  13600
5   이상   날개  1936  18300
6   이상   권태  1937  14100


In [16]:
# Q2 - (b)

def categorize(n: int) -> str:
    if n < 10000:
        return '짧음'
    elif n <= 15000:
        return '보통'
    else:
        return '긴'

works2['분량'] = works2['글자수'].apply(categorize)
print(works2[['작품', '글자수', '분량']])

           작품    글자수  분량
0          봄봄  12500  보통
1         동백꽃   9800  짧음
2         만무방  13600  보통
3     운수 좋은 날  11200  보통
4   B사감과 러브레터   7400  짧음
5          날개  18300   긴
6          권태  14100  보통
7     벙어리 삼룡이  16800   긴
8        물레방아  14500  보통
9    레디메이드 인생  22000   긴
10   메밀꽃 필 무렵  11800  보통


In [17]:
# Q2 - (c)

print(works2.sort_values(by=['연도', '글자수'], ascending=[True, False]))

     작가         작품    연도    글자수  분량
3   현진건    운수 좋은 날  1924  11200  보통
7   나도향    벙어리 삼룡이  1925  16800   긴
8   나도향       물레방아  1925  14500  보통
4   현진건  B사감과 러브레터  1925   7400  짧음
9   채만식   레디메이드 인생  1934  22000   긴
2   김유정        만무방  1934  13600  보통
0   김유정         봄봄  1935  12500  보통
5    이상         날개  1936  18300   긴
10  이효석   메밀꽃 필 무렵  1936  11800  보통
1   김유정        동백꽃  1936   9800  짧음
6    이상         권태  1937  14100  보통


In [18]:
# Q4 - (a)

full = pd.merge(works2, authors, on='작가', how='left')
print(full)
print(full.shape)

"""how='left'는 왼쪽 표(works2)의 모든 행을 유지하고
오른쪽 표(authors)에서 키가 일치하는 행을 붙이는 방식이기 때문에,
따라서 authors에만 있고 works2에는 없는 염상섭은 full에 포함되지 않는다."""

     작가         작품    연도    글자수  분량    생년    몰년
0   김유정         봄봄  1935  12500  보통  1908  1937
1   김유정        동백꽃  1936   9800  짧음  1908  1937
2   김유정        만무방  1934  13600  보통  1908  1937
3   현진건    운수 좋은 날  1924  11200  보통  1900  1943
4   현진건  B사감과 러브레터  1925   7400  짧음  1900  1943
5    이상         날개  1936  18300   긴  1910  1937
6    이상         권태  1937  14100  보통  1910  1937
7   나도향    벙어리 삼룡이  1925  16800   긴  1902  1926
8   나도향       물레방아  1925  14500  보통  1902  1926
9   채만식   레디메이드 인생  1934  22000   긴  1902  1950
10  이효석   메밀꽃 필 무렵  1936  11800  보통  1907  1942
(11, 7)


"how='left'는 왼쪽 표(works2)의 모든 행을 유지하고\n오른쪽 표(authors)에서 키가 일치하는 행을 붙이는 방식이기 때문에,\n따라서 authors에만 있고 works2에는 없는 염상섭은 full에 포함되지 않는다."